# Optimized Inference Deployment

In this section we will explore advanced frameworks for optimizing LLM deployments: Text Generation Inference (TGI), vLLM, and llama.cpp. These applications are primarily used in production environments to serve LLMs to users. This section focuses on how to deply these frameworks in production rather than how to use them for inference on a single machine. 

## Framework Selection Guide

TGI, vLLM, and llama.cpp serve similar purposes but have distinct characteristics that make them better suited for different use cases. Let's look at the key differences between them, focusing on performance and integration.

### Memory Management and Performance

**TGI** is designed to be stable and predictable in production, using fixed sequence lengths to keep memory usage consistent. 

TGI manages memory uses Flash Attention 2 and continuous batching techniques. This means it can process attention calculations very efficiently and keep the GPU busy by constantly feeding it work. 

The system can move parts of the model between CPU and GPU when needed, which helps handle larger models.

Flash Attention is a technique that optimizes the attention mechanism in transformer models by addressing memory bandwidth bottlenecks.

The attention mechanism has quadratic complexity and memory usage, making it inefficient for long sequences.

The key innovation is in how it manages memory transfers between High Bandwith Memory (HBM) and faster SRAM cache.

Traditional attention repeatedly transfers data between HBM and SRAM, creating bottlenecks by leaving the GPU idle. Flash Attention loads data once into SRAM and performs all calculations there, minimizing expensive memory transfers.

While the benefits are most significant during training, Flash Attention's reduced VRAM usage and improved efficiency make it valuable for inference as well, enabling faster and more scalable LLM serving.

**vLLM** takes a different approach by using PagedAttention. Just like how a computer manages its memory in pages, vLLM spluts the model's memory into smaller blocks. This clever system means it can handle different-sized requests more flexibly and doesn't waste memory space. It's particularly good at sharing memory between different requests and reduces memory fragmentation, which makes the whole system more efficient.

PagedAttention is a technique that addresses another critical bottleneck in LLM inference: KV cache memory management.

During text generation, the model stores attention keys and values (KV cache) for ach generated token to reduce redundant computations. The KV cache can become enormous, especially with long sequences or multiple concurrent requests.

vLLM's key innovation lies in how it manages this cache:
1. **MemoryPaging:** Instead of treating the KV cache as one large block, it is divided into fixed-size "pages" (similar to virtual memory in operating systems).
2. **Non-contiguous Storage:** Pages don't need to be stored contiguously in GPU memory, allowing for more flexible memory allocation.
3. **Page Table Management:** A page table tracks which pages belong to which sequence, enabling efficient lookup and access.
4. **Memory Sharing:** For operations like parallel sampling, pages storing the KV cache for the prompt can be shared across multiple sequences.

The PageAttentin approach can lead up to 24x higher-throughput compared to traditional methods, making it a game-changer for production LLM deployments.


**llama.cpp** is a highly optimized C/C++ implementation originally designed for running LLaMA models on consumer hardware. It focuses on CPU efficiency with optional GPU acceleration and is ideal for resource-constrained environments.

llama.cpp uses quantization techniques to reduce model size and memory requirements while maintaining good performance. It implements optimized kernels for various CPU architectures and supports basic KV cache management for efficient token generation.

Quantization in llama.cpp reduces the precision of model weights from 32-bit or 16-but floating point to lower precision formats like 8-bit integers (INT8), 4-biy, or even lower. This significantly reduces memory usage and improves inference speed with minial quality loss.

Key quantization features in llama.cpp include:

1. **Multiple Quantization Levels:** Supports 8-bit, 4-bit, 3-bit, and even 2-bit quantization
2. **GGML/GGUF Format:** Uses custom tensor formats optimized for quantized inference
3. **Mixed Precision:** Can apply different quantization levels to different parts of the model
4. **Hardware-Specific Optimizations:** Includes optimized code paths for various CPU architectures (AVX2, AVX512, NEON)

This approach enables running billion-parameter models to consumer hardware with limited memory, making it pperfect for local deployments and edge devices.

## Deployment and Integration

Let's move on to the deployment and integration differences between the frameworks.

**TGI** excels in enterprise-level deployment with its production-ready features. It comes with built-in Kubernetes support and includes everything you need for running in production, like monitoring through Prometheus and Grafana, automatic scaling, and comprehensive saftey features. The system also includes enterprise-grade logging and various protective measures like content filtering and rate limiting to keep your deployment secure and stable.

**vLLM** takes a more flexible, developer-friendly approach to deployment. It's built with Python at its core and can easily replace OpenAI's API in your exisiting applications. The framework focuses on delivering raw performance and can be customized to fit your specific needs. It works particularly well with Ray for managing clusters, making it a great choice when you need high performance and adaptability.

**llama.cpp** prioritizes simplicity and portability. Its server implementation is lightweight and can run on a widge range of hardware, from powerful servers to consumer laptops and even some high-end mobile devices. With minimal dependencies and a simple C/C++ core, it's easy to deploy in environments where installing Python frameworks would be challenging. The server provides an OpenAI-compatible API while mantaining a much smaller resource footprint than other solutions.

### Getting Started

Let's explore how to use these frameworks for deploying LLMs, starting with installation and basic setup.

#### Installation and Basic Setup

##### llama.cpp

llama.cpp is easy to install and use, requiring minimal dependencies and supporting both CPU and GPU inference.

First, install and build llama.cpp

```
# Download CMake
brew install cmake

# Clone the repository
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# Build the project
cmake -B build
cmake --build build --config Release

# Download the SmolLM2-1.7B-Instruct-GGUF model
curl -L -O https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF/resolve/main/smollm2-1.7b-instruct-q4_k_m.gguf
```

Then, launch the server (with OpenAI API compatibility):
```
# Start the server
llama-server -m smollm2-1.7b-instruct-q4_k_m.gguf --port 8080
```

In [2]:
from huggingface_hub import InferenceClient

# Init client pointing to llama.cpp server
client = InferenceClient(
    model="http://localhost:8080//v1/chat/completions", # URL to the llama.cpp server
    token="sk-no-key-required", # llama.pp server requires this placeholder
)
# For chat format
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)

print(f'Text Generation Response:\n {response.choices[0].message.content}')

Text Generation Response:
 Once upon a time, in a magical forest, there lived a curious little rabbit named Rosie. Rosie was known for her adventurous spirit and her love for exploring the vast and beautiful forest. She was also known for her unique ability to speak with animals.

One sunny morning, Rosie decided she wanted to learn more about the forest. She packed a small bag with some snacks and set off on a new adventure. As she journeyed deeper into the forest, she came across a wise


#### vLLM

vLLM is easy to install and use, with both OpenAI API compatibility and a native Python interface.

First, launch the vLLM OpenAI-compatible server:
```
# Create a python3.12 virtual env
/opt/homebrew/bin/python3.12 -m venv vllm-env 

# Clone vLLM git
git clone https://github.com/vllm-project/vllm.git

# install torch torchvision
pip install torch torchvision

# install all dependencies in for directory (vllm)
pip install -e .

#
python -m vllm.entrypoints.openai.api_server \
  --model HuggingFaceTB/SmolLM2-360M-Instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --max-model-len 2048
```

In [7]:
from huggingface_hub import InferenceClient

# Initialize client pointing to vLLM endpoint
client = InferenceClient(
    model="http://localhost:8000/v1",  # URL to the vLLM server
)

# For chat format
response = client.chat_completion(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)

Once upon a time, in a land far, far away, there lived a kind-hearted and wise old man named Gideon. Gideon was a farmer who had spent his life tending to his land and nurturing the land to grow good and healthy crops. He was also a very skilled healer, and he was always happy to help his friends in need.

One day, while Gideon was out in his field, he noticed that his beloved chicken coop had been damaged


## Basic Text Generation

### llama.cpp

For llama.cpp, you can set advanced parameters when launching the server:

```
llama-server -m smollm2-1.7b-instruct-q4_k_m.gguf -c 4096 --port 8080 --batch-size 512
```

*Batch size* is used for prompt evaluation

*c* Context Size

In [2]:
from huggingface_hub import InferenceClient

client = InferenceClient(model="http://localhost:8080/v1", token="sk-no-key-required")

# Advanced parameters examples
response = client.chat_completion(
    messages=[
        {"role": "system", "content": "You are a creative storyteller."},
        {"role": "user", "content": "Write a creative story"},
    ],
    temperature=0.8,
    max_tokens=200,
    top_p=0.95
)

print(response.choices[0].message.content)


In a world where the skies were painted with hues of crimson and gold, and the stars shone with a soft, ethereal glow, there lived a young girl named Lila. She was an ordinary girl, with an extraordinary spirit and a heart full of wonder. Lila lived in a village surrounded by vast, sprawling forests, shimmering lakes, and towering mountains, where mythical creatures roamed free.

One day, while wandering through the forest, Lila stumbled upon a hidden glade. In the center of the clearing, she found a magnificent tree, its trunk as wide as a castle, its branches stretching up to the stars. The tree was adorned with a crown of shimmering leaves and flowers that shimmered like diamonds.

As Lila approached the tree, she felt an unusual energy emanating from it. A soft, pulsing glow enveloped her, and she began to see visions of the past and the future. She saw a great calamity that would


You can also use the llama.cpp native library for even more control:

In [5]:
# Using llama-cpp-python package for direct model access
from llama_cpp import Llama

# Load the model

llm = Llama(
    model_path="smollm2-1.7b-instruct-q4_k_m.gguf",
    n_ctx=4096,  # Context window size
    n_threads=8,  # CPU threads
    n_gpu_layers=0,  # GPU layers (0 = CPU only)
)

# Format prompt according to the mode;'s expected format
prompt = """<|im_start|>system
You are a creative storyteller.
<|im_end|>
<|im_start|>user
Write a creative story
<|im_end|>
<|im_start|>assistant
"""

output = llm(
    prompt,
    max_tokens=200,
    temperature=0.8,
    top_p=0.95,
    frequency_penalty=0.5,
    presence_penalty=0.5,
    stop=["<|im_end|>"],
)

print(output["choices"][0]["text"])

llama_model_load_from_file_impl: using device Metal (Apple M4 Pro) - 16378 MiB free
llama_model_loader: loaded meta data with 34 key-value pairs and 218 tensors from smollm2-1.7b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Smollm2 1.7B 8k Mix7 Ep2 v2
llama_model_loader: - kv   3:                            general.version str              = v2
llama_model_loader: - kv   4:                       general.organization str              = Loubnabnl
llama_model_loader: - kv   5:                           general.finetune str              = 8k-mix7-ep2
llama_model_loader: - kv   6:                           ge

In a world where time was currency, the rich could live forever and the poor were left with nothing but the tick-tock of their mortality. A young man named Jack was born with an unusual gift – he could manipulate time. With his powers, Jack could slow down or speed up time to suit his needs.

One day, while working at a small clock repair shop, Jack's life changed forever when a wealthy businessman walked in and asked for his help fixing an antique pocket watch. To Jack's surprise, the watch was priceless and held a secret. It was created by a mysterious timekeeper who had imbued it with the ability to manipulate time itself.

As Jack worked on the watch, he began to realize that he could use its power to change his life forever. He could speed up time when he needed it, slowing down the days until their arrival like a gentle breeze on a summer's day. And when his life became too slow, he could speed it up,
